In [0]:

# Configuracion
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

CATALOG = "banco"
SCHEMA = "banking_analytics"

# Tablas de origen
SILVER_TABLES = {
    "branches": f"{CATALOG}.{SCHEMA}.silver_branches",
    "customers": f"{CATALOG}.{SCHEMA}.silver_customers",
    "accounts": f"{CATALOG}.{SCHEMA}.silver_accounts",
    "transactions": f"{CATALOG}.{SCHEMA}.silver_transactions"
}

# Tablas destino Gold
GOLD_TABLES = {
    "dim_customer": f"{CATALOG}.{SCHEMA}.dim_customer",
    "dim_account": f"{CATALOG}.{SCHEMA}.dim_account",
    "dim_branch": f"{CATALOG}.{SCHEMA}.dim_branch",
    "dim_date": f"{CATALOG}.{SCHEMA}.dim_date",
    "fact_transactions": f"{CATALOG}.{SCHEMA}.fact_transactions"
}


In [0]:
# Cargar tablas silver
df_silver_branches = spark.table(SILVER_TABLES["branches"])

df_silver_customers = spark.table(SILVER_TABLES["customers"])

df_silver_accounts = spark.table(SILVER_TABLES["accounts"])

df_silver_transactions = spark.table(SILVER_TABLES["transactions"])

print("Branches:", df_silver_branches.count())
print("Customers:", df_silver_customers.count())
print("Accounts:", df_silver_accounts.count())
print("Transactions:", df_silver_transactions.count())


In [0]:
# Verificación de Schema
print("CUSTOMERS")
df_silver_customers.printSchema()

print("ACCOUNTS")
df_silver_accounts.printSchema()

print("BRANCHES")
df_silver_branches.printSchema()

print("TRANSACTIONS")
df_silver_transactions.printSchema()


In [0]:
# Construccion de la dimension Customer

df_dim_customer = (
    df_silver_customers

    .select(
        "customer_id",
        "customer_name",
        "birth_date",
        "country",
        "registration_date",
        "email",
        "customer_segment"
    )

    # Generar clave sustituta determinista
    .withColumn(
        "customer_key",
        F.xxhash64(F.col("customer_id"))
    )

    # Organizar las columnas
    .select(
        "customer_key",
        "customer_id",
        "customer_name",
        "birth_date",
        "country",
        "registration_date",
        "email",
        "customer_segment"
    )
)

display(df_dim_customer.limit(10))


In [0]:

# Validar la dimension Customer

total_silver = df_silver_customers.count()

total_dimension = df_dim_customer.count()

duplicate_customers = (
    df_dim_customer
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

duplicate_keys = (
    df_dim_customer
    .groupBy("customer_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Clientes Silver:", total_silver)
print("Clientes DIM:", total_dimension)
print("Customer ID duplicados:", duplicate_customers)
print("Customer Key duplicados:", duplicate_keys)

assert total_silver == total_dimension
assert duplicate_customers == 0
assert duplicate_keys == 0

print("Validaciones de DIM_CUSTOMER exitosas.")


In [0]:
# GUARDAR TABLA DIMENSION CUSTOMER
(
    df_dim_customer.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_TABLES["dim_customer"])
)

In [0]:
# Validar persistencia de datos
df_dim_customer_saved = spark.table(
    GOLD_TABLES["dim_customer"]
)

print("Registros guardados:", df_dim_customer_saved.count())

display(
    df_dim_customer_saved.limit(10)
)

In [0]:
# Construcción de Dimen Branch
df_dim_branch = (
    df_silver_branches
    .select(
        "branch_id",
        "branch_name",
        "city",
        "region",
        "country",
        "opening_date",
        "is_active"
    )
    .withColumn(
        "branch_key",
        F.xxhash64(F.col("branch_id"))
    )
    .select(
        "branch_key",
        "branch_id",
        "branch_name",
        "city",
        "region",
        "country",
        "opening_date",
        "is_active"
    )
)

display(df_dim_branch.limit(10))

In [0]:
# Validar Integridad de datos
total_silver = df_silver_branches.count()
total_dimension = df_dim_branch.count()

duplicate_branches = (
    df_dim_branch
    .groupBy("branch_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

duplicate_keys = (
    df_dim_branch
    .groupBy("branch_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Agencias Silver:", total_silver)
print("Agencias DIM:", total_dimension)
print("Branch ID duplicados:", duplicate_branches)
print("Branch Key duplicados:", duplicate_keys)

assert total_silver == total_dimension
assert duplicate_branches == 0
assert duplicate_keys == 0

print("Validaciones de DIM_BRANCH exitosas.")

In [0]:
#Persistir tabla
(
    df_dim_branch.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_TABLES["dim_branch"])
)

In [0]:
# Validar persistencia
df_dim_branch_saved = spark.table(
    GOLD_TABLES["dim_branch"]
)

print(
    "Registros guardados:",
    df_dim_branch_saved.count()
)

display(df_dim_branch_saved.limit(10))

In [0]:
# cargar dimensiones
df_dim_customer = spark.table(
    GOLD_TABLES["dim_customer"]
)

df_dim_branch = spark.table(
    GOLD_TABLES["dim_branch"]
)

In [0]:
# Construir la dimension mediante joins

df_dim_account = (
    df_silver_accounts.alias("a")

    .join(
        df_dim_customer.select(
            "customer_id",
            "customer_key"
        ).alias("c"),
        F.col("a.customer_id") == F.col("c.customer_id"),
        "inner"
    )

    .join(
        df_dim_branch.select(
            "branch_id",
            "branch_key"
        ).alias("b"),
        F.col("a.branch_id") == F.col("b.branch_id"),
        "inner"
    )

    .select(
        F.xxhash64(F.col("a.account_id")).alias("account_key"),
        F.col("a.account_id"),
        F.col("c.customer_key"),
        F.col("b.branch_key"),
        F.col("a.account_type"),
        F.col("a.balance"),
        F.col("a.currency"),
        F.col("a.opening_date"),
        F.col("a.account_status")
    )
)

display(df_dim_account.limit(10))

In [0]:
# Validacion de integridad de datos

total_silver = df_silver_accounts.count()
total_dimension = df_dim_account.count()

duplicate_accounts = (
    df_dim_account
    .groupBy("account_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

duplicate_keys = (
    df_dim_account
    .groupBy("account_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

null_customer_keys = (
    df_dim_account
    .filter(F.col("customer_key").isNull())
    .count()
)

null_branch_keys = (
    df_dim_account
    .filter(F.col("branch_key").isNull())
    .count()
)

print("Cuentas Silver:", total_silver)
print("Cuentas DIM:", total_dimension)
print("Account ID duplicados:", duplicate_accounts)
print("Account Key duplicados:", duplicate_keys)
print("Customer Key nulos:", null_customer_keys)
print("Branch Key nulos:", null_branch_keys)

In [0]:
# Persistencia de datos
(
    df_dim_account.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_TABLES["dim_account"])
)

In [0]:
# Validar persistencia de datos
df_dim_account_saved = spark.table(
    GOLD_TABLES["dim_account"]
)

print(
    "Registros guardados:",
    df_dim_account_saved.count()
)

display(df_dim_account_saved.limit(10))

In [0]:
# Indetificar Rango de Fechas
df_transaction_dates = (
    df_silver_transactions
    .select(
        F.to_date("transaction_date").alias("transaction_date")
    )
)

date_range = df_transaction_dates.agg(
    F.min("transaction_date").alias("min_date"),
    F.max("transaction_date").alias("max_date")
)

display(date_range)

In [0]:
# Zona horaria
print(
    "Spark session timezone:",
    spark.conf.get("spark.sql.session.timeZone")
)

In [0]:
# Generación de calendario
date_limits = date_range.first()

min_date = date_limits["min_date"]
max_date = date_limits["max_date"]

print("Fecha inicial:", min_date)
print("Fecha final:", max_date)

In [0]:
df_dim_date = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                F.lit(min_date),
                F.lit(max_date),
                F.expr("INTERVAL 1 DAY")
            )
        ).alias("full_date")
    )
)

display(df_dim_date.limit(10))

In [0]:
# construir los atributos de calendario
df_dim_date = (
    df_dim_date

    # Clave sustituta de fecha
    .withColumn(
        "date_key",
        F.date_format("full_date", "yyyyMMdd").cast("integer")
    )

    # Componentes de fecha
    .withColumn("day", F.dayofmonth("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("quarter", F.quarter("full_date"))
    .withColumn("year", F.year("full_date"))

    # Nombre del mes
    .withColumn(
        "month_name",
        F.element_at(
            F.array(
                F.lit("January"), F.lit("February"),
                F.lit("March"), F.lit("April"),
                F.lit("May"), F.lit("June"),
                F.lit("July"), F.lit("August"),
                F.lit("September"), F.lit("October"),
                F.lit("November"), F.lit("December")
            ),
            F.col("month")
        )
    )

    # Día de la semana: 1 = Monday, 7 = Sunday
    .withColumn(
        "day_of_week",
        F.pmod(F.dayofweek("full_date") + 5, F.lit(7)) + 1
    )

    # Nombre del día
    .withColumn(
        "day_name",
        F.element_at(
            F.array(
                F.lit("Monday"), F.lit("Tuesday"),
                F.lit("Wednesday"), F.lit("Thursday"),
                F.lit("Friday"), F.lit("Saturday"),
                F.lit("Sunday")
            ),
            F.col("day_of_week")
        )
    )

    # Semana del año
    .withColumn(
        "week_of_year",
        F.weekofyear("full_date")
    )

    # Indicador de fin de semana
    .withColumn(
        "is_weekend",
        F.col("day_of_week").isin(6, 7)
    )

    # Orden final de columnas
    .select(
        "date_key",
        "full_date",
        "day",
        "month",
        "month_name",
        "quarter",
        "year",
        "day_of_week",
        "day_name",
        "week_of_year",
        "is_weekend"
    )
)

display(df_dim_date.limit(15))

In [0]:
# validar dimension date
total_dates = df_dim_date.count()

duplicate_dates = (
    df_dim_date
    .groupBy("full_date")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

duplicate_date_keys = (
    df_dim_date
    .groupBy("date_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

null_dates = (
    df_dim_date
    .filter(
        F.col("full_date").isNull() |
        F.col("date_key").isNull()
    )
    .count()
)

print("Total de fechas:", total_dates)
print("Fechas duplicadas:", duplicate_dates)
print("Date Keys duplicadas:", duplicate_date_keys)
print("Fechas o claves nulas:", null_dates)

assert total_dates == 669
assert duplicate_dates == 0
assert duplicate_date_keys == 0
assert null_dates == 0

print("Validaciones de DIM_DATE exitosas.")

In [0]:
# Validar coberturas de fecha
df_transaction_dates_distinct = (
    df_silver_transactions
    .select(
        F.to_date("transaction_date").alias("full_date")
    )
    .distinct()
)

df_missing_dates = (
    df_transaction_dates_distinct
    .join(
        df_dim_date.select("full_date"),
        on="full_date",
        how="left_anti"
    )
)

missing_dates = df_missing_dates.count()

print("Fechas distintas en transacciones:",
      df_transaction_dates_distinct.count())

print("Fechas sin correspondencia en DIM_DATE:",
      missing_dates)

assert missing_dates == 0

print("Validación de cobertura temporal exitosa.")

In [0]:
# Guardar dimension date
(
    df_dim_date.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_TABLES["dim_date"])
)

In [0]:
# Validar persistencia
df_dim_date_saved = spark.table(
    GOLD_TABLES["dim_date"]
)

print("Registros guardados:", df_dim_date_saved.count())

display(
    df_dim_date_saved.orderBy("full_date").limit(10)
)

In [0]:
# Cargar dimensiones
df_dim_customer = spark.table(GOLD_TABLES["dim_customer"])
df_dim_account = spark.table(GOLD_TABLES["dim_account"])
df_dim_branch = spark.table(GOLD_TABLES["dim_branch"])
df_dim_date = spark.table(GOLD_TABLES["dim_date"])

print("Dimensiones cargadas correctamente.")

In [0]:
df_fact_transactions = (
    df_silver_transactions.alias("t")

    # Relacionar la transacción con su cuenta
    .join(
        df_dim_account.select(
            "account_id",
            "account_key",
            "customer_key",
            "branch_key"
        ).alias("a"),
        F.col("t.account_id") == F.col("a.account_id"),
        "left"
    )

    # Relacionar la transacción con su fecha
    .join(
        df_dim_date.select(
            "full_date",
            "date_key"
        ).alias("d"),
        F.to_date(F.col("t.transaction_date")) == F.col("d.full_date"),
        "left"
    )

    # Seleccionar las columnas de la tabla de hechos
    .select(
        F.col("t.transaction_id"),
        F.col("a.customer_key"),
        F.col("a.account_key"),
        F.col("a.branch_key"),
        F.col("d.date_key"),
        F.col("t.transaction_date"),
        F.col("t.amount"),
        F.col("t.currency"),
        F.col("t.transaction_type"),
        F.col("t.channel"),
        F.col("t.status"),
        F.lit(1).cast("integer").alias("transaction_count")
    )
)

display(df_fact_transactions.limit(10))

In [0]:
# Validar consistencia de datos
total_silver = df_silver_transactions.count()
total_fact = df_fact_transactions.count()

duplicate_transactions = (
    df_fact_transactions
    .groupBy("transaction_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

null_transaction_ids = (
    df_fact_transactions
    .filter(F.col("transaction_id").isNull())
    .count()
)

null_account_keys = (
    df_fact_transactions
    .filter(F.col("account_key").isNull())
    .count()
)

null_customer_keys = (
    df_fact_transactions
    .filter(F.col("customer_key").isNull())
    .count()
)

null_branch_keys = (
    df_fact_transactions
    .filter(F.col("branch_key").isNull())
    .count()
)

null_date_keys = (
    df_fact_transactions
    .filter(F.col("date_key").isNull())
    .count()
)

print("Transacciones Silver:", total_silver)
print("Transacciones Fact:", total_fact)
print("Transacciones duplicadas:", duplicate_transactions)
print("Transaction ID nulos:", null_transaction_ids)
print("Account Key nulos:", null_account_keys)
print("Customer Key nulos:", null_customer_keys)
print("Branch Key nulos:", null_branch_keys)
print("Date Key nulos:", null_date_keys)

In [0]:
# Creando la tabla delta de fact_transactions
(
    df_fact_transactions.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_TABLES["fact_transactions"])
)

In [0]:
# Validando persistencia

df_fact_transactions_saved = spark.table(
    GOLD_TABLES["fact_transactions"]
)

total_saved = df_fact_transactions_saved.count()

print("Registros en Fact:", total_saved)

assert total_saved == 97709

print("Persistencia de FACT_TRANSACTIONS exitosa.")

display(df_fact_transactions_saved.limit(10))

In [0]:
# Datos foraneos

df_fact = spark.table(GOLD_TABLES["fact_transactions"])

df_dim_account = spark.table(GOLD_TABLES["dim_account"])
df_dim_customer = spark.table(GOLD_TABLES["dim_customer"])
df_dim_branch = spark.table(GOLD_TABLES["dim_branch"])
df_dim_date = spark.table(GOLD_TABLES["dim_date"])

# Validar ACCOUNT_KEY
orphan_accounts = (
    df_fact.select("account_key").distinct()
    .join(
        df_dim_account.select("account_key"),
        on="account_key",
        how="left_anti"
    )
    .count()
)

# Validar CUSTOMER_KEY
orphan_customers = (
    df_fact.select("customer_key").distinct()
    .join(
        df_dim_customer.select("customer_key"),
        on="customer_key",
        how="left_anti"
    )
    .count()
)

# Validar BRANCH_KEY
orphan_branches = (
    df_fact.select("branch_key").distinct()
    .join(
        df_dim_branch.select("branch_key"),
        on="branch_key",
        how="left_anti"
    )
    .count()
)

# Validar DATE_KEY
orphan_dates = (
    df_fact.select("date_key").distinct()
    .join(
        df_dim_date.select("date_key"),
        on="date_key",
        how="left_anti"
    )
    .count()
)

print("Account Keys huérfanas:", orphan_accounts)
print("Customer Keys huérfanas:", orphan_customers)
print("Branch Keys huérfanas:", orphan_branches)
print("Date Keys huérfanas:", orphan_dates)

assert orphan_accounts == 0
assert orphan_customers == 0
assert orphan_branches == 0
assert orphan_dates == 0

print("Auditoría de integridad referencial exitosa.")